In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from matplotlib.colors import to_rgb
from matplotlib.colors import LinearSegmentedColormap

# Juptyer magic: For export. Makes the plots size right for the screen 
%matplotlib inline
# %config InlineBackend.figure_format = 'retina'

%config InlineBackend.figure_formats = ['svg'] 


torch.backends.cudnn.deterministic = True
seed = np.random.randint(1,200)
# seed = 107 #59
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
print(seed)
g = torch.Generator()
g.manual_seed(seed)

subfolder = 'node_2d_new'
import os
if not os.path.exists(subfolder):
    os.makedirs(subfolder)
    


In [ ]:



# Model Params
#Import of the model dynamics that describe the neural ODE
#The dynamics are based on the torchdiffeq package, that implements ODE solvers in the pytorch setting
from models.neural_odes import NeuralODEvar

#for neural ODE based networks the network width is constant. In this example the input is 2 dimensional
hidden_dim, data_dim = 2, 2
output_dim = 1

#T is the end time of the neural ODE evolution, time_steps are the amount of discretization steps for the ODE solver
T, time_steps = 20, 20 #
step_size = T/time_steps
num_params = 10 #the number of distinct parameters present in the interval. they are spread equidistant over the interval [0, T].

layers_hidden = 0 #the amount of layers in the vector field (these layers do not correspond to time discretization but to the expressivity of the vector field)

non_linearity = 'tanh' #'relu' #
architecture = 'inside' #outside




# Training Params
cross_entropy = True #True supported with binary classification only
num_epochs = 300

seed = np.random.randint(1,200)
print(seed)



# ResNet Parameters and Training Data

In [ ]:
import models.training
from models.training import make_circles_uniform



# Generate training data
n_points = 4000 #number of points in the dataset
batch_size = 100

inner_radius = 0.5
outer_radius = 1
buffer = 0.2

plotrange = [-2.5,2.5]

import importlib
importlib.reload(models.training) # Reload the module

train_loader, test_loader = make_circles_uniform(output_dim = output_dim, n_samples = n_points, inner_radius = inner_radius, outer_radius = outer_radius, buffer = buffer, cross_entropy = cross_entropy, batch_size = batch_size, seed = seed, filename = subfolder + '/circles_dataset')

# nODE Model Params

In [ ]:
from models.training import doublebackTrainer


# seed = 16


torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
node = NeuralODEvar(device, data_dim, hidden_dim, output_dim = output_dim, non_linearity=non_linearity, 
                    architecture=architecture, T=T, time_steps=time_steps, num_params = num_params, layers_hidden = layers_hidden, cross_entropy=cross_entropy)


optimizer_node = torch.optim.Adam(node.parameters(), lr=1e-3) 
trainer_node = doublebackTrainer(node, optimizer_node, device, cross_entropy=cross_entropy) 



In [ ]:
from plots.plots import classification_levelsets
from IPython.display import Image

footnote = node.footnote()

num_cycles = 3
epochs_cycle = int(300/3)

for i in range(num_cycles):
    trainer_node.train(train_loader, epochs_cycle)
    
    fig_name_base = os.path.join(subfolder, 'levelsets' + str(i))
    classification_levelsets(node, fig_name_base, footnote = footnote, plotlim= [-2,2])

for i in range(num_cycles):
    fig_name_base = os.path.join(subfolder, 'levelsets' + str(i))
    img = Image(filename = fig_name_base + '.png', width = 400)
    display(img)

In [ ]:
import plots.plots
importlib.reload(plots.plots) # Reload the module to ensure the latest changes are applied

from plots.plots import classification_levelsets
from IPython.display import Image

footnote = f'{num_epochs = }, {cross_entropy = }, {num_params = },\n {time_steps = }, {output_dim = }, {hidden_dim = }, {seed = }'
        
fig_name_base = os.path.join(subfolder, 'levelsets')
classification_levelsets(node, fig_name_base, footnote = footnote, plotlim= [-2,2])
img1 = Image(filename = fig_name_base + '.png', width = 400)
display(img1)
plt.plot(trainer_node.histories['epoch_loss_history'])
plt.xlim(0, len(trainer_node.histories['epoch_loss_history']) - 1)
plt.ylim(0)
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.show()

# XOR data set

In [ ]:
from models.training import make_xor

train_loader, test_loader = make_xor(1)

In [ ]:
# Model Params
num_hidden = 6 # number of hidden layers. The total network has additionl 2 layers: input to hidden and hidden to output
input_dim = 2
hidden_dim = 2
output_dim = 1
activation = 'tanh' #'relu' and 'tanh' are supported

# Training Params
load_file = None
cross_entropy = True #True supported with binary classification only
num_epochs = 300



In [ ]:
footnote = f'num_hidden={num_hidden}, hidden_dim={hidden_dim}, output_dim={output_dim}, act={activation}, seed={seed}, ce={cross_entropy}'

model_base, acc_base, losses_base = train_until_threshold(ResNet,
    train_loader, test_loader,
    load_file = load_file, max_retries=5, threshold=0.95,
    input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_hidden=num_hidden, skip_param=0, activation=activation
)

plot_loss_curve(losses_base, title=f"Base Model Loss Curve", filename = 'xor_ff6hidden')

In [ ]:
X_test, y_test = next(iter(test_loader))
plot_decision_boundary(model_base, X_test, y_test, show=True, file_name= 'ff6hiddenxor' + str(num_hidden), footnote = footnote, amount_levels= 100)
plot_level_sets(model_base, show=True, file_name= 'ff6hiddenxor_contour' + str(num_hidden), footnote = footnote, amount_levels= 20)

In [ ]:
hidden_dim = 3
num_hidden = 2

footnote = f'num_hidden={num_hidden}, hidden_dim={hidden_dim}, output_dim={output_dim}, act={activation}, seed={seed}, ce={cross_entropy}'

model_wide, acc_wide, losses_wide = train_until_threshold(ResNet,
    train_loader, test_loader,
    load_file = load_file, max_retries=5, threshold=0.95,
    input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_hidden=num_hidden, skip_param=0, activation=activation
)

plot_loss_curve(losses_wide, title=f"Wide Model Loss Curve", filename = 'ffxorloss')

In [ ]:
X_test, y_test = next(iter(test_loader))
plot_decision_boundary(model_wide, X_test, y_test, show=True, file_name= 'ffxor' + str(num_hidden), footnote = footnote, amount_levels= 100)
plot_level_sets(model_wide, show=True, file_name= 'ffxor_contour' + str(num_hidden), footnote = footnote, amount_levels= 30)